<a href="https://colab.research.google.com/github/Pranayshukla0610/MLOPS-Project/blob/main/MLOPS_On_ANN_On_MNIST_Fashion_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf project

In [2]:
import os

folders = [
    "project",
    "project/src",
    "project/data",
    "project/models",
    "project/logs",
    "project/artifacts",
    "project/config"
]

for folder in folders:
  os.makedirs(folder, exist_ok=True)

print("Folder Structure Created")

Folder Structure Created


In [3]:
open("project/src/__init__.py","w").close()
print('src is now a package')

src is now a package


In [4]:
%%writefile project/src/data_loader.py

import tensorflow as tf
import os
import numpy as np

def load_data():
  (X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
  os.makedirs('project/data', exist_ok = True)

  np.save('project/data/X_train.npy', X_train)
  np.save('project/data/y_train.npy', y_train)
  np.save('project/data/X_test.npy', X_test)
  np.save('project/data/y_test.npy',y_test)

  return X_train, X_test, y_train, y_test

Writing project/src/data_loader.py


In [5]:
%%writefile project/src/eda.py

import matplotlib.pyplot as plt
import seaborn as sns

def perform_eda(X_train,y_train):
  print("Shape:",X_train.shape)

  plt.figure(figsize=(20,16))
  for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(str(y_train[i]))
    plt.axis("off")

  plt.show()

Writing project/src/eda.py


In [6]:
%%writefile project/src/preprocessing.py

def preprocess_data(X_train,y_train):
  X_train = X_train/255.0
  X_test = X_test/255.0

  X_train = X_train.reshape(X_train.shape[0],-1)
  X_test = X_test.reshape(X_test.shape[0],-1)

  return X_train, X_test

Writing project/src/preprocessing.py


In [7]:
%%writefile project/src/model.py

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

def build_model(input_dim):
  model = Sequential()

  model.add(Dense(256, activation='relu', input_dim=input_dim))
  model.add(Dropout(0.3))

  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.3))

  model.add(Dense(64, activation='relu'))
  model.add(Dropout(0.3))

  model.add(Dense(10, activation='softmax'))

  model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

  return model

Writing project/src/model.py


In [8]:
%%writefile project/src/train.py

from src.data_loader import load_data
from src.preprocessing import preprocess_data
from src.model import build_model

Writing project/src/train.py


In [10]:
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
import os

def train_model():
  X_train, X_test, y_train, y_test = load_data()
  X_train, X_test = preprocess_data(X_train, X_test)
  model = build_model(X_train.shape[1])
  os.makedirs("project/logs", exist_ok = True)

  callbacks = [
      EarlyStopping(patience=5, restore_best_weights=True),
      CSVLogger("project/logs/log.csv")
  ]

  history = model.fit(
      X_train, y_train,
      epochs=True,
      batch_size=128,
      validation_split=0.2,
      callbacks = callbacks,
      verbose = 1
  )

  os.makedirs("project/models", exist_ok =True)
  model.save("project/models/model.h5")

  return model, history, X_test, y_test

In [11]:
%%writefile project/src/evaluate.py
from sklearn.metrics import classification_report
import numpy as np

def evaluate_model(model, X_test, y_test):
  y_pred = model.predict(X_test)
  y_pred = np.argmax(y_pred, axis=1)
  print(classification_report(y_test,y_pred))

Writing project/src/evaluate.py


In [12]:
%%writefile project/src/plot.py

def plot_history(history):
  plt.plot(history.history['accuracy'], label='Train')
  plt.plot(history.history['val_accuracy'],label = 'Val')
  plt.legend()
  plt.title("Accuracy")
  plt.show()

Writing project/src/plot.py


In [13]:
%%writefile project/main.py

import sys
import os

sys.path.append(os.path.abspath('project'))

from src.train import train_model
from src.evaluate import evaluate_model
from src.plot import plot_history

def main():
  model, history, X_test, y_test = train_model()
  plot_history(history)
  evaluate_model(model, X_test, y_test)

if __name__ = "__main__":
  main()

Writing project/main.py
